# ANN Predictions — Equinox and Solstice Days

This notebook evaluates the ANN trained on `FINAL_MASTER.csv` against the **held-out equinox and solstice observations** stored in `ann_evals/data/equinox_solstice.csv`.

Because these dates were excluded from `FINAL_MASTER.csv` during preprocessing, this is a **true out-of-distribution evaluation** — the model has never seen data from these days.

## Days Covered

| DOY | Event |
|-----|-------|
| 79 | March Equinox |
| 172 | June Solstice |
| 266 | September Equinox |
| 356 | December Solstice |

## Prerequisite

`ann_evals/outputs/ann_full_model.pt` must exist. Run `train_full.py` (or the `scenario1_full.ipynb` notebook) first if it does not.

## Outputs
- `ann_evals/outputs/predictions_ann_equinox_solstice.csv`
- `ann_evals/outputs/ann_equinox_solstice_scatter.png`

## 1. Imports and Path Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import warnings
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings("ignore")

In [ ]:
# Resolve paths relative to this notebook (ann_evals/notebooks/)
notebook_dir = Path().resolve()                      # ann_evals/notebooks/
evals_dir    = notebook_dir.parent                   # ann_evals/
data_dir     = evals_dir / "data"
output_dir   = evals_dir / "outputs" / "scenario5_equinox_solstice"
output_dir.mkdir(parents=True, exist_ok=True)

DATA_CSV   = data_dir  / "equinox_solstice.csv"
CHECKPOINT = evals_dir / "outputs" / "scenario1_full" / "ann_full_model.pt"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Data file  :", DATA_CSV,   "| exists:", DATA_CSV.exists())
print("Checkpoint :", CHECKPOINT, "| exists:", CHECKPOINT.exists())
print("Device     :", DEVICE)

In [ ]:
FEATURES  = ["DayOfYear", "Hour", "Longitude", "Latitude", "F10.7"]
TARGET    = "foF2"

STATIONS       = ["CP", "ElginAB", "Jicamarca", "MilstonHill", "Ramey"]
STATION_COLOR  = dict(zip(STATIONS, ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]))
STATION_MARKER = dict(zip(STATIONS, ["o", "s", "^", "D", "v"]))

DOY_LABEL = {
    79:  "Mar Equinox (DOY 79)",
    172: "Jun Solstice (DOY 172)",
    266: "Sep Equinox (DOY 266)",
    356: "Dec Solstice (DOY 356)",
}

## 2. Load Checkpoint

The checkpoint bundles:
- `model_state_dict` — trained weights
- `train_mean` / `train_std` — z-score normalisation statistics computed on `FINAL_MASTER.csv` training split
- `features` — ordered feature list used during training
- `epoch` — best epoch (early stopping)

Carrying the normalisation stats in the checkpoint means inference is fully self-contained — no access to the original training data is needed.

In [ ]:
if not CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {CHECKPOINT}\n"
        "Run train_full.py (or scenario1_full.ipynb) first."
    )

ck = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=False)
train_mean = ck["train_mean"]
train_std  = ck["train_std"]

print(f"Best epoch : {ck['epoch'] + 1}")
print(f"Features   : {ck['features']}")
print()
print("Normalisation stats (from training split):")
for f, m, s in zip(ck["features"], train_mean, train_std):
    print(f"  {f:<12}  mean={m:.4f}  std={s:.4f}")

## 3. Rebuild and Load the ANN

The architecture must match what was used in `train_full.py` exactly:
`Input(5) → 16 → 32 → 64 → 128 → 64 → 32 → 16 → Output(1)` with ReLU activations and a linear output.

In [ ]:
class ANNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(5,   16), nn.ReLU(),
            nn.Linear(16,  32), nn.ReLU(),
            nn.Linear(32,  64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64,  32), nn.ReLU(),
            nn.Linear(32,  16), nn.ReLU(),
            nn.Linear(16,   1),
        )
    def forward(self, x):
        return self.net(x)

model = ANNModel().to(DEVICE)
model.load_state_dict(ck["model_state_dict"])
model.eval()

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Model loaded and set to eval mode.")

## 4. Load Equinox/Solstice Data

In [ ]:
df       = pd.read_csv(DATA_CSV)
X        = df[FEATURES].values.astype(np.float32)
y        = df[TARGET].values.astype(np.float32)
stations = df["Station"].values

print(f"Loaded : {DATA_CSV.name}")
print(f"Shape  : {df.shape}")
print(f"Stations: {sorted(df['Station'].unique())}")
print(f"DOYs   : {sorted(df['DayOfYear'].unique())}")
df.head()

## 5. Normalise and Run Inference

Apply the same z-score transform that was computed on the training split of `FINAL_MASTER.csv`.

In [ ]:
X_norm = ((X - train_mean) / train_std).astype(np.float32)

with torch.no_grad():
    y_pred = model(torch.from_numpy(X_norm).to(DEVICE)).cpu().numpy().flatten()

print(f"Inference complete. Predictions shape: {y_pred.shape}")
print(f"Predicted foF2 range: {y_pred.min():.3f} – {y_pred.max():.3f} MHz")
print(f"Observed  foF2 range: {y.min():.3f} – {y.max():.3f} MHz")

## 6. Overall Metrics

In [ ]:
mse  = float(mean_squared_error(y, y_pred))
rmse = float(np.sqrt(mse))
mae  = float(mean_absolute_error(y, y_pred))
r2   = float(r2_score(y, y_pred))
bias = float(np.mean(y_pred - y))

print("Overall metrics (all stations, all equinox/solstice days):")
print(f"  MSE  = {mse:.4f}")
print(f"  RMSE = {rmse:.4f} MHz")
print(f"  MAE  = {mae:.4f} MHz")
print(f"  R²   = {r2:.4f}")
print(f"  Bias = {bias:+.4f} MHz")
print(f"  N    = {len(y):,}")

## 7. Per-DOY Metrics

Breaking down accuracy by event shows whether the model generalises equally well to equinoxes and solstices, or whether it struggles with a particular season.

In [ ]:
per_doy_rows = []
for doy in sorted(df["DayOfYear"].unique()):
    mask   = df["DayOfYear"].values == doy
    yt, yp = y[mask], y_pred[mask]
    per_doy_rows.append({
        "DOY":   doy,
        "Event": DOY_LABEL.get(doy, f"DOY {doy}"),
        "RMSE":  round(float(np.sqrt(mean_squared_error(yt, yp))), 4),
        "MAE":   round(float(mean_absolute_error(yt, yp)), 4),
        "R2":    round(float(r2_score(yt, yp)), 4),
        "Bias":  round(float(np.mean(yp - yt)), 4),
        "N":     int(mask.sum()),
    })

pd.DataFrame(per_doy_rows).set_index("DOY")

## 8. Per-Station Metrics

In [ ]:
per_stn_rows = []
for stn in STATIONS:
    mask = stations == stn
    if not mask.any():
        continue
    yt, yp = y[mask], y_pred[mask]
    per_stn_rows.append({
        "Station": stn,
        "RMSE":    round(float(np.sqrt(mean_squared_error(yt, yp))), 4),
        "MAE":     round(float(mean_absolute_error(yt, yp)), 4),
        "R2":      round(float(r2_score(yt, yp)), 4),
        "Bias":    round(float(np.mean(yp - yt)), 4),
        "N":       int(mask.sum()),
    })

pd.DataFrame(per_stn_rows).set_index("Station")

## 9. Scatter Plot (Observed vs Predicted)

Points are coloured by station. The dashed diagonal is the 1:1 line — perfect predictions lie on it.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

for stn in STATIONS:
    mask = stations == stn
    if not mask.any():
        continue
    ax.scatter(y[mask], y_pred[mask], s=20, alpha=0.6,
               color=STATION_COLOR[stn], marker=STATION_MARKER[stn], label=stn)

lims = [min(y.min(), y_pred.min()) - 0.3, max(y.max(), y_pred.max()) + 0.3]
ax.plot(lims, lims, "k--", lw=1)
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("Observed foF2 (MHz)")
ax.set_ylabel("Predicted foF2 (MHz)")
ax.set_title("Scenario 5 — Equinox & Solstice Prediction", fontsize=10)
ax.legend(fontsize=8, markerscale=1.4)

txt = (f"RMSE={rmse:.3f}  MAE={mae:.3f}\n"
       f"R²={r2:.4f}   Bias={bias:+.3f}\nN={len(y):,}")
ax.text(0.03, 0.97, txt, transform=ax.transAxes, va="top", fontsize=8,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))

plt.tight_layout()
scatter_path = output_dir / "ann_equinox_solstice_scatter.png"
plt.savefig(scatter_path, dpi=150)
plt.show()
print(f"Saved: {scatter_path}")

## 10. Save Predictions CSV

In [ ]:
out_df = df[["Station", "DayOfYear", "Hour", "Longitude", "Latitude", "F10.7", "foF2"]].copy()
out_df.rename(columns={"foF2": "foF2_obs"}, inplace=True)
out_df["foF2_pred"] = y_pred.round(4)
out_df["residual"]  = (out_df["foF2_pred"] - out_df["foF2_obs"]).round(4)

pred_path = output_dir / "predictions_ann_equinox_solstice.csv"
out_df.to_csv(pred_path, index=False)
print(f"Saved: {pred_path}  ({len(out_df):,} rows)")
out_df.head(10)